In [ ]:
######################################
# importing core and 3rd-party modules
######################################
import os
from pathlib import Path
from itertools import cycle
import numpy as np
from scipy.stats import alpha
import matplotlib.pyplot as plt

###############################################
# Move to project root to easily import modules
###############################################
try:  # avoid changes if already set
    print("Working from: ", PROJECT_ROOT)
except NameError:
    try:  # running from Spyder
        PROJECT_ROOT = Path(__file__).resolve()
    except NameError:  # running as Jupyter N•otebook
        # PROJECT_ROOT = "D:\people\Idomic\gSTED-sFCS" # System PC (room -110)
        # PROJECT_ROOT = "E:\gSTED-sFCS" # Oleg's office
        PROJECT_ROOT = "/Users/ido.michealovich/Documents/GitHub/Personal/gSTED-sFCS" # Mac Laptop
#         PROJECT_ROOT = "D:\MEGA\BioPhysics_Lab\Optical_System\gSTEDsFCS"
    os.chdir(PROJECT_ROOT)
    print("Working from: ", PROJECT_ROOT)
    
from utilities.display import Plotter
from utilities.file_utilities import search_database
from data_analysis.notebook_analysis_tools import SolutionSFCSExperimentHandler
from utilities.helper import Limits
from data_analysis.polymer_physics import gyradius, plot_theoretical_structure_factor_in_ax, debye_structure_factor_fit, casassa_structure_factor_fit
from data_analysis.correlation_function import SolutionSFCSExperiment, combine_measurements_list

##################################################
# Setting up data paths and other global constants
##################################################
# DATA_ROOT = Path("D:\OneDrive - post.bgu.ac.il\gSTED_sFCS_Data")  # Laptop/Lab PC (same path)
# DATA_ROOT = Path(r"E:\OneDrive - post.bgu.ac.il\gSTED_sFCS_Data")  # Oleg's office PC
DATA_ROOT = Path(r"/Users/ido.michealovich/Library/CloudStorage/OneDrive-post.bgu.ac.il/gSTED_sFCS_Data")  # Mac Laptop
# DUMP_ROOT =  Path(r"/Users/ido.michealovich/tmp/Personal")  # Mac Laptop

# Interactive plotting for Jupyter Notebook 7
%matplotlib widget

################
# Function Defs
################
# SHOULD_SAVE_FIGURES = True
SHOULD_SAVE_FIGURES = False

def save_figure(fig, fig_path):
    if SHOULD_SAVE_FIGURES:
        fig.savefig(fig_path)
        print(f"Figure saved at: '{fig_path}'")
    else:
        print(f"Figure was NOT saved! 'SHOULD_SAVE' is False!")

Find relvant measurements templates in `DATA_ROOT` (if needed)

In [ ]:
print(search_database(DATA_ROOT, ["alignment"]))

# Figure - Angular Scan Image showing bright spots (before/after)
# CANCELED - NOTHING TO SHOW

Loading and processing a single file from the main measurement, with and without "bright pixel alleviation"

In [ ]:
# DATA_TYPE = "solution"
#
# general_options = dict(
#     should_load_data=False, # if False, will skip loading & processing raw data (or even checking if it exists).
#
#     was_processed=False,
#
#     force_save = False,
# )
#
# data_configs = {
#     "With 'Bright Pixel Removal'": dict(
#         confocal_date="20_03_2024",
#         confocal_template="YOYO30kbp_OC_1200nW_angular_exc_123355_*.pkl",
#         confocal_kwargs={"file_selection": "Use 1"},
#
#         sted_date="20_03_2024",
#         sted_template="YOYO30kbp_OC_3uW_Gated2ns_angular_sted_260mW_141955_*.pkl",
#         sted_kwargs={"file_selection": "Use 1"},
#
#         force_processing=True,
#
#         should_alleviate_bright_pixels=True, # KEY (and only) DIFFERENCE
#
#         **general_options,
#     ),
#     "Without 'Bright Pixel Removal'": dict(
#         confocal_date="20_03_2024",
#         confocal_template="YOYO30kbp_OC_1200nW_angular_exc_123355_*.pkl",
#         confocal_kwargs={"file_selection": "Use 1"},
#
#         sted_date="20_03_2024",
#         sted_template="YOYO30kbp_OC_3uW_Gated2ns_angular_sted_260mW_141955_*.pkl",
#         sted_kwargs={"file_selection": "Use 1"},
#
#         force_processing=True,
#         **general_options,
#     ),
# }
#
# # ############################
# # Load/Process the experiments
# # ############################
# # FORCE_PROCESSING = False
# FORCE_PROCESSING = True
#
# # PRINT_LOG_FILES = True
# PRINT_LOG_FILES = False
#
# exp_handler = SolutionSFCSExperimentHandler(
#     data_root=DATA_ROOT,
#     force_processing=FORCE_PROCESSING,
# )
# exp_handler.load_experiments(
#     data_configs,
#     print_log_files=PRINT_LOG_FILES
# )

Comparing the scan images

In [ ]:
# for label, exp in exp_handler.exp_dict.items():
#     exp.sted.display_scan_images(2)

In [ ]:
# exp_handler.plot_correlation_functions()

# Figure - Comparison of theoretical structure factors

In [ ]:
def debye_structure_factor_fit_approx(q, Rg: float) -> np.ndarray:
    x = (q * Rg) ** 2
    if (q <= 1/Rg).all():
        return 1 - (q * Rg)**2 / 3
    elif (q >= 1/Rg).all():
        return 2 / (q * Rg)**2
    else:
        raise ValueError(f"This should never happen!")

def casassa_structure_factor_fit_approx(q, Rg: float) -> np.ndarray:
    if (q <= 1/Rg).all():
        # same as debye
        return 1 - (q * Rg)**2 / 3
    elif (q >= 1/Rg).all():
        # half than debye
        return 1 / (q * Rg)**2
    else:
        raise ValueError(f"This should never happen!")


# Create a q array to use
q = np.linspace(1, 30, 100_000)

# Set the gyradius
r_g_linear_um = 0.250
r_g_ring_um = 0.250

# Get the theoreticals
s_q_linear = debye_structure_factor_fit(q, r_g_linear_um, 1)
s_q_ring = casassa_structure_factor_fit(q, r_g_ring_um, 1)

# Get the small q approximation
q_under_Rg_linear = q[Limits(0, 0.75/r_g_linear_um).valid_indices(q)]
s_q_linear_small_q = debye_structure_factor_fit_approx(q_under_Rg_linear, r_g_linear_um)

# Get the high q approximations
r_g_factor = 5
q_over_Rg_linear = q[Limits(r_g_factor / r_g_linear_um, np.inf).valid_indices(q)]
s_q_linear_high_q = debye_structure_factor_fit_approx(q_over_Rg_linear, r_g_linear_um)

q_over_Rg_ring = q[Limits(r_g_factor / r_g_ring_um, np.inf).valid_indices(q)]
s_q_ring_high_q = casassa_structure_factor_fit_approx(q_over_Rg_ring, r_g_ring_um)

# Plot
with Plotter(
    figsize=(6, 4),
    x_scale="log",
    y_scale="log",
) as ax:
    # plot the structure factors
    lines_debye = ax.plot(q, s_q_linear, label="Debye Structure Factor")
    ax.plot(q_over_Rg_linear, s_q_linear_high_q, "--", color=lines_debye[0].get_color(), label="_Large q approx.: $2/(qR_g)^2$")

    lines_casassa = ax.plot(q, s_q_ring, label="Casassa Structure Factor")
    ax.plot(q_over_Rg_ring, s_q_ring_high_q, "--", color=lines_casassa[0].get_color(), label="_Large q approx.: $1/(qR_g)^2$")

    # plot the small q approximation
    ax.plot(q_under_Rg_linear, s_q_linear_small_q, "--", color="black", label="_Small q approx.: $1-(qR_g)^2/3$")

    # Set ax attributes
    ax.set_xlabel("$q,\ \mu m$")
    ax.set_ylabel("$S(q)$")

    ax.legend()

# Save
save_figure(ax.figure, "theoreticalStructureFactors.eps")

# Figure - YOYO-1 lifetime as a function of labeling density

Here's how I found the relevant experiments

In [ ]:
# print(search_database(DATA_ROOT, ["1in", "exc", "YOYO", "300"]))
print(search_database(DATA_ROOT, ["1in", "exc", "YOYO"]))

Load the relevenat experiments

In [ ]:
DATA_TYPE = "solution"

general_options = dict(
    should_load_data=False, # if False, will skip loading & processing raw data (or even checking if it exists).

    was_processed=False,

    should_parallel_process=True,

    force_save = True,
    # force_save = False,
)

data_configs = {
    "$\lambda=1:100$": dict(
        confocal_date="17_07_2023",
        confocal_template="YOYO33kbp_1in100_TE_angular_exc_190140_*.pkl",

        force_processing=False,
        **general_options,
    ),
    "$\lambda=1:20$": dict(
        confocal_date="27_09_2023",
        confocal_template="YOYO300bp_1in20_20uW_angular_exc_164021_*.pkl",

        force_processing=False,
        **general_options,
    ),
    "$\lambda=1:15$": dict(
        confocal_date="31_08_2023",
        confocal_template="YOYO300bp_1in15_TE_20uW_angular_exc_162459_*.pkl",

        force_processing=False,
        **general_options,
    ),
    "$\lambda=1:10$": dict(
        confocal_date="31_08_2023",
        confocal_template="YOYO300bp_1in10_TE_20uW_angular_exc_161448_*.pkl",

        force_processing=False,
        **general_options,
    ),
    "$\lambda=1:5$": dict(
        confocal_date="28_08_2023",
        confocal_template="YOYO300bp_1in5_fresh_angular_exc_155230_*.pkl",

        force_processing=False,
        **general_options,
    ),
}

# ############################
# Load/Process the experiments
# ############################
FORCE_PROCESSING = False
# FORCE_PROCESSING = True

# PRINT_LOG_FILES = True
PRINT_LOG_FILES = False

exp_handler = SolutionSFCSExperimentHandler(
    data_root=DATA_ROOT,
    force_processing=FORCE_PROCESSING,
)
exp_handler.load_experiments(
    data_configs,
    print_log_files=PRINT_LOG_FILES
)

Now get the lifetime for each experiment

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

label2lt = {}

with Plotter(
    figsize=(8, 5),
    y_scale="log",
    xlim=(3, 75),
    ylim=(6e-3, 1.05),
) as ax:
    for label, exp in exp_handler.exp_dict.items():
        meas = exp.confocal
        hist = meas.tdc_calib.all_hist_norm
        t = meas.tdc_calib.t_hist

        # Clean up NaNs or infinities
        valid_mask = np.isfinite(hist)
        t = t[valid_mask]
        hist = hist[valid_mask]

        h_max, j_max = hist.max(), hist.argmax()
        t_max = t[j_max]

        # Fit the lifetime
        fit_params = meas.tdc_calib.fit_lifetime_hist(
            fit_range=Limits(t_max + 0.1, 95),
            fit_param_estimate=(h_max, 4, h_max * 1e-3),
        )

        # Store the fitted lifetime
        label2lt[label] = fit_params.beta["tau"], fit_params.beta_error["tau"]

        # Plot the histogram (normalized)
        ys = fit_params.ys / fit_params.ys.max()
        fitted_y = fit_params.fitted_y / fit_params.fitted_y.max()
        line = ax.plot(fit_params.xs, ys, "o", markersize=2, alpha=0.1)
        ax.plot(
            fit_params.x,
            fitted_y,
            "--",
            label=label,
            color=line[0].get_color(),
        )

    # axes properties
    ax.set_xlabel("Time after excitation pulse, ns")
    ax.set_ylabel("Photon Counts, Normalized")

    # Move the legend to the upper left so it does not overlap the inset
    ax.legend(loc="upper right")

# Create an inset axes in the top-right corner
inset_ax = inset_axes(ax, width="35%", height="35%", loc="upper center")

# Same internal "density" map, but we will relabel the x ticks
densities = [1/100, 1/20, 1/15, 1/10, 1/5]
density_map = {label: density for label, density in zip(label2lt.keys(), densities)}

densities = []
lifetimes = []
lifetime_errors = []
for lbl, (tau, tau_error) in label2lt.items():
    if lbl in density_map:
        densities.append(density_map[lbl])
        lifetimes.append(tau)
        lifetime_errors.append(tau_error)

# Plot density vs. lifetime on the inset
inset_ax.plot(densities, lifetimes, marker="o", linestyle="-")
inset_ax.errorbar(
    densities,
    lifetimes,
    lifetime_errors,
    fmt="none",
    label="_Error",
    elinewidth=0.5,
    zorder=2,
)

# inset_ax properties
inset_ax.set_xlabel("$\lambda,\ bp$")
inset_ax.set_ylabel("$\\tau,\ ns$")

# Convert the fractional densities into bp labels for x-ticks
# (e.g., 1/150 on the axis should display "150" as the tick label).
all_densities = sorted(density_map.values())  # for a clean left-to-right scale
inset_ax.set_xticks(all_densities)
inset_ax.set_xticklabels(
    [f"1:{int(1 / d)}" for d in all_densities]
)
# tilt the ticks by angle
inset_ax.tick_params(labelrotation=45)

# Save
save_figure(ax.figure, "YOYO-1Lifetime.eps")

# Main Experiments

In [ ]:
DATA_TYPE = "solution"

general_options = dict(
    should_load_data=False, # if False, will skip loading & processing raw data (or even checking if it exists).

    was_processed=False,

    should_parallel_process=True,

    byte_data_slice=slice(21, None, None), # ignore the first 100 elements of data - massive jumps in runtime there sometimes

    force_save = True,
    # force_save = False,
)

data_configs = {
    # FIXME: the "FL SD L 1" is somewhat problematic in that it cannot be fit the screened SF without the dilute experiment, which is a bad measurement
    # "FL SD L 1": dict(
    #     confocal_date="20_03_2024",
    #     confocal_template="YOYO30kbp_L_1200nW_angular_exc_131240_*.pkl",
    #
    #     sted_date="20_03_2024",
    #     sted_template="YOYO30kbp_L_3uW_Gated2ns_angular_sted_260mW_155323_*.pkl",
    #
    #     force_processing=False,
    #     # force_processing=True,
    #
    #     **general_options,
    # ),
    "FL SD OC 1": dict(
        confocal_date="20_03_2024",
        confocal_template="YOYO30kbp_OC_1200nW_angular_exc_123355_*.pkl",
        confocal_kwargs={"file_selection": "Don't Use 4"},

        sted_date="20_03_2024",
        sted_template="YOYO30kbp_OC_3uW_Gated2ns_angular_sted_260mW_141955_*.pkl",

        force_processing=False,
        **general_options,
    ),
    # FIXME: the "FL SD OC 2" is somewhat problematic in that it cannot be fit the screened SF without the dilute experiment, which is a bad measurement
    # "FL SD OC 2": dict(
    #     confocal_date="15_04_2024",
    #     confocal_template="YOYO_30kbp_OC_1200uW_snake_angular_exc_202814_*.pkl",
    #     confocal_kwargs={"file_selection": "Don't Use 4, 10, 16, 22, 28, 34, 40, 46, 52, 58, 64, 70, 76, 82, 83"},
    #     # NOTES: 1% background obsereved
    #
    #     sted_date="15_04_2024",
    #     sted_template="YOYO_30kbp_OC_2700uW_snake_angular_sted_260mW_205639_*.pkl",
    #     sted_kwargs={"file_selection": "Don't Use 18, 24, 30, 36, 42, 84, 53, 54, 60, 66, 72, 73, 78, 84, 90, 96, 102, 108, 114, 120, 126, 132, 138, 144, 150"},
    #
    #     force_processing=False,
    #     **general_options,
    # ),
    "FL D OC 1": dict(
        confocal_date="27_03_2024",
        confocal_template="YOYO30kbp_OC_Dilute_7uW_angular_exc_123502_*.pkl",
        confocal_kwargs={"file_selection": "Don't Use 4"},

        sted_date="25_03_2024",
        sted_template="YOYO30kbp_OC_Dilute_13uW_angular_sted_260mW_174839_*.pkl",
        sted_kwargs={"file_selection": "Don't Use 1, 45"},

        force_processing=False,
        **general_options,
    ),
    # NOTE - Added after the fact
    "FL D SC": dict(
        confocal_date="21_03_2024",
        confocal_template="YOYO30kbp_SC_1200nW_Diluted3X_angular_exc_133712_*.pkl",

        sted_date="21_03_2024",
        sted_template="YOYO30kbp_SC_3uW_Gated2ns_angular_sted_260mW_140058_*.pkl",

        force_processing=False,
        **general_options,
    ),
    "FL D L 2": dict(
        confocal_date="16_04_2024",
        confocal_template="YOYO_30kbp_Dilute_L_13uW_snake_angular_exc_104229_*.pkl",
        confocal_kwargs={"file_selection": "Don't Use 3, 6, 8, 19, 22, 24, 27, 32, 37, 40, 47, 51, 66, 70, 81, 86, 91, 93, 94, 101, 107, 116"},

        sted_date="16_04_2024",
        sted_template="YOYO_30kbp_Dilute_L_13uW_snake_angular_sted_260mW_114035_*.pkl",

        force_processing=False,
        **general_options,
    ),
}

# ############################
# Load/Process the experiments
# ############################
FORCE_PROCESSING = False
# FORCE_PROCESSING = True

# PRINT_LOG_FILES = True
PRINT_LOG_FILES = False

exp_handler = SolutionSFCSExperimentHandler(
    data_root=DATA_ROOT,
    force_processing=FORCE_PROCESSING,
)
exp_handler.load_experiments(
    data_configs,
    print_log_files=PRINT_LOG_FILES
)

# Figure - angular scanning image example

In [ ]:
conf_meas = exp_handler.exp_dict["FL SD OC 1"].confocal

# Get the actual approximate dimensions of the image from the scan settings
line_length_um = conf_meas.scan_settings["max_line_length_um"]
line_shift_um = conf_meas.scan_settings["line_shift_um"]
n_lines = conf_meas.scan_settings["n_lines"]
samples_per_line = conf_meas.scan_settings["samples_per_line"]

# scale of "true" image
height_um = n_lines * line_shift_um
width_um = line_length_um
pixel_width_um = width_um / samples_per_line

# rotate the 3D stack such that the first axis is the image index
conf_imgs = np.moveaxis(conf_meas.scan_images_dstack, -1, 0)

# Plot the Nth scan image with its ROI
N = 0
with Plotter(figsize=(8, 6)) as ax:
    # Show the image and store the reference
    im = ax.imshow(
        conf_imgs[N],
        extent=[0, width_um, 0, height_um],
        interpolation="none",
        origin="upper",
        aspect="equal",
    )

    # Create the colorbar below the axis, horizontally
    cbar = plt.colorbar(im, ax=ax, shrink=0.7, orientation="horizontal", location="bottom")
    cbar.set_label("Number of Photons", fontsize=12)

    # Overlay your ROIs
    sec_roi_list = conf_meas.rois[N]
    for sec_roi in sec_roi_list:
        col_px = np.array(sec_roi["col"])
        row_px = np.array(sec_roi["row"])
        roi_x = col_px / samples_per_line * width_um
        roi_y = (n_lines - row_px) / n_lines * height_um + line_shift_um / 2
        ax.plot(roi_x, roi_y, color="white", lw=1.5)

    # Axis labels
    ax.set_xlabel("$x,~\\mu m$")
    ax.set_ylabel("$y,~\\mu m$")

print(
    f"The scan image is created from photon collected during {conf_meas.data[0].general.duration_s:.0f} seconds, "
    f"At a (linear) speed of {conf_meas.scan_settings['speed_um_s'] * 1e-3:.1f} mm/s. "
    f"Each pixel represents an area of approximately {line_shift_um} um x {pixel_width_um} um "
    f"(={line_shift_um * pixel_width_um:.1f} um^2)"
)

# print(conf_meas.scan_settings)
# Save
save_figure(ax.figure, "scanImage.eps")

### Figure - Comparison of detector-gating vs. post-measurement gating

In [ ]:
# Set filters
exp_handler.exact_filter = ["FL D OC 1", "FL SD OC 1"]

with Plotter(
    subplots=(2, 2),
    figsize=(8, 8),
    xlim=(1e-3, 0.8),
    ylim=(5e-2, 1.05),
    # sharey=True,  # Adjust as needed (you can also try sharex=False if desired)
) as axes:
    # Unpack axes into two rows
    (post_gated_ax, hard_gated_ax), (post_gated_ax_log, hard_gated_ax_log) = axes

    # Loop over experiments and plot
    for label, exp in exp_handler.filtered_exp_dict.items():
        if "SD" in label:
            # Top row: original scale
            exp.plot_correlation_functions(parent_ax=hard_gated_ax, show_non_tdc_gated=False)
            # Bottom row: log/linear scales
            exp.plot_correlation_functions(
                parent_ax=hard_gated_ax_log,
                x_scale="log",
                y_scale="linear",
                show_non_tdc_gated=False,
            )
        else:
            # Top row: original scale
            exp.plot_correlation_functions(parent_ax=post_gated_ax, show_non_tdc_gated=False)
            # Bottom row: log/linear scales
            exp.plot_correlation_functions(
                parent_ax=post_gated_ax_log,
                x_scale="log",
                y_scale="linear",
                show_non_tdc_gated=False,
            )

    # Set axes properties for the top row
    post_gated_ax.set_title("Dilute\n(post-measurement gating)")
    hard_gated_ax.set_title("Semi-Dilute\n(live gating)")
    post_gated_ax.set_xlabel("")
    hard_gated_ax.set_xlabel("")
    hard_gated_ax.set_ylabel("")

    # Set axes properties for the bottom row (log/linear)
    # post_gated_ax_log.set_title("Dilute\n(post-measurement gating, log/linear)")
    # hard_gated_ax_log.set_title("Semi-Dilute\n(live gating, log/linear)")
    post_gated_ax_log.set_xlabel("$r,~\\mu m$")
    hard_gated_ax_log.set_xlabel("$r,~\\mu m$")
    hard_gated_ax_log.set_ylabel("")

    # Manually adjust the legend for each axis
    for row in axes:
        for ax in row:
            ax.legend(["Confocal", "gSTED"])

# Specifically set the limits differently for the log/linear plots
post_gated_ax_log.set_xlim(1e-2, 1.5)
hard_gated_ax_log.set_xlim(1e-2, 1.5)
post_gated_ax_log.set_ylim(0, 1.05)
hard_gated_ax_log.set_ylim(0, 1.05)

# Label the axes for the caption
labels = ['(a)', '(b)', '(c)', '(d)']
for ax, label in zip([post_gated_ax, hard_gated_ax,
                      post_gated_ax_log, hard_gated_ax_log],
                     labels):
    # Place the label in the upper-left corner of each subplot
    ax.text(0.05, 0.1, label, transform=ax.transAxes,
            fontsize='medium',
            va='top', ha='left')

# Save figure if needed
# save_figure(axes[0][0].figure, "ACFsAndIntersection.eps")


# Figure - Comparison of dilute and semi-dilute ACFs and Hankel transforms (same as for calibration graph, but for the samples)

In [ ]:
# # Plot the spatial ACFs in real and inverse (q) space, side-by-side
# with Plotter(super_title="$33~kbp$ Relaxed, Non-Concatenated Plasmids", subplots=(1, 2), figsize=(8, 4)) as axes:
#     ax_real, ax_inverse = axes
#
#     # plot the ACFs and their transforms
#     for label, exp in exp_handler.exp_dict.items():
#         print(f"Processing {label}")
#         exp.plot_correlation_functions(parent_ax=ax_real)
#         for cf_name, cf in exp.cf_dict.items():
#             cf.hankel_transforms["gaussian"].plot(parent_ax=ax_inverse, plot_interpolations=False)
#
#     # Set the axis properties for the figure
#     ax_real.set_title("Spatial ACFs")
#     ax_real.set_xlabel("$r,~\mu m$")
#     ax_real.set_xlim(0, 0.45)
#     ax_real.set_ylim(3e-1, 1.05)
#
#     ax_inverse.set_title("Inverse-Space ACFs")
#     ax_inverse.set_xlabel("$q,~\mu m^{-1}$")
#     ax_inverse.set_ylabel("")
#     ax_inverse.set_xlim(2, 17)
#     ax_inverse.set_ylim(1e-2, 1.01)
#
#     # legends
#     ax_real.legend([])
#     ax_inverse.legend(
#         [
#             "Semi-Dilute: Confocal",
#             "Semi-Dilute: STED",
#             "Semi-Dilute: gSTED",
#             "Dilute: Confocal",
#             "Dilute: STED",
#             "Dilute: gSTED",
#         ],
#         # loc="upper right"
#     )
#
# # Save
# save_figure(ax_real.figure, "SampleRealInverse.eps")

In [ ]:
with Plotter(super_title="$33~kbp$ Relaxed, Non-Concatenated Plasmids", figsize=(8, 4)) as ax:
    for i, (label, exp) in enumerate(exp_handler.exp_dict.items()):
        print(f"Processing {label}")

        # Plot the correlation functions, capturing the lines that get created
        lines = exp.plot_correlation_functions(
            parent_ax=ax,
            show_non_tdc_gated=False,
            y_scale="linear",
            x_scale="log",
            x_field="vt_um",
        )

        # We assume two lines come out of plot_correlation_functions:
        #   lines[0] -> confocal
        #   lines[1] -> gSTED
        # Force first to be blue, second to be orange
        lines[0].set_color("tab:blue")
        lines[1].set_color("tab:orange")

        # Semi-dilute => dashed lines, dilute => solid lines
        # (Adjust the condition to match how your labels are defined.)
        if " SD " in label:
            lines[0].set_linestyle("--")
            lines[1].set_linestyle("--")
        elif "SC" in label:
            lines[0].set_linestyle("-.")
            lines[1].set_linestyle("-.")
        elif " L " in label:
            lines[0].set_linestyle(":")
            lines[1].set_linestyle(":")
        else:
            lines[0].set_linestyle("-")
            lines[1].set_linestyle("-")

    # Now set axes properties once for the whole figure
    ax.set_xlabel("$r, \mu m$")
    ax.set_ylabel("$ACF$")
    ax.set_xlim(2e-2, 3)
    ax.set_ylim(-0.05, 1.05)

    # Legend
    ax.legend(
        [
            "Semi-Dilute, Nicked: Confocal",
            "Semi-Dilute, Nicked: gSTED",
            "Dilute, Nicked: Confocal",
            "Dilute, Nicked: gSTED",
            "Dilute, Supercoiled: Confocal",
            "Dilute, Supercoiled: gSTED",
            "Dilute, Linearized: Confocal",
            "Dilute, Linearized: gSTED",
        ],
        # loc="upper right",
    )

    # Save
    # save_figure(ax.figure, "SampleLag.eps")


In [ ]:
raise RuntimeError

# Figure - Comparison of dilute and semi-dilute structure factors of relaxed, non-concatenated long plasmids (confocal and gated-STED)

### Fit The Structure Factors

In [ ]:
# Filter
exp_handler.exact_filter = ["FL D OC 1"]

# Fit the SFs
confocal_max_q = 9.2
sted_max_q = 13

exp_handler.fit_structure_factors(
    "confocal",
    x_limits=Limits(0, confocal_max_q),
)
exp_handler.fit_structure_factors(
    "sted",
    x_limits=Limits(0, sted_max_q),
)

# Filter
exp_handler.exact_filter = ["FL SD OC 1"]

# Fit the SFs
confocal_max_q = 11.5
sted_max_q = 13.2

exp_handler.fit_structure_factors(
    "confocal",
    x_limits=Limits(0, confocal_max_q),
)
exp_handler.fit_structure_factors(
    "sted",
    x_limits=Limits(0, sted_max_q),
)

Plot Structure Factors

In [ ]:
# Set filters
# exp_handler.exact_filter = ["FL D OC 1"]
# exp_handler.exact_filter = ["FL SD OC 1"]
exp_handler.exact_filter = ["FL D OC 1", "FL SD OC 1"]

marker_cycle = cycle(["o", "^", "s", "D", "v", "p", "*"])
with Plotter(
    figsize=(6, 4),
    xlim=(2, 18),
    ylim=(2e-2, 1.05),
) as ax:
    for label, exp in exp_handler.exp_dict.items():
        exp.plot_structure_factors(show_non_tdc_gated=False, parent_ax=ax, marker=next(marker_cycle))

    # Plot q dependencies of ideal and FG polymers
    q = exp.confocal.cf["confocal"].structure_factors["gaussian"].q
    coeff_ideal = 3
    coeff_fg = 3.5
    q_ideal = q[Limits(5/coeff_ideal, 10/coeff_ideal).valid_indices(q)]
    plot_theoretical_structure_factor_in_ax(ax, q_ideal, coeff=coeff_ideal, model="ideal")
    q_fg = q[Limits(5/coeff_fg, 10/coeff_fg).valid_indices(q)]
    plot_theoretical_structure_factor_in_ax(ax, q_fg, coeff=coeff_fg, model="fractal globule")

    # Manually adjust the legend
    ax.set_title("Structure Factors")
    ax.legend(
        [
            "Semi-dilute, Nicked", "_",
            "Semi-dilute (gSTED), Nicked", "_",
            "Dilute, Nicked", "_",
            "Dilute (gSTED), Nicked", "_",
            "Dilute, Supercoiled",
            "Dilute (gSTED), Supercoiled",
            "Ideal polymer ($q^{-2}$)",
            "Fractal globule ($q^{-3}$)",
        ]
    )

# Print the gyration radius estimates for a linearized and nicked 30kbp DNA plasmid
print("Gyradius estimates:")
print("Linearized: ", gyradius(30_000, "linear", "gaussian"))
print("Nicked: ", gyradius(30_000, "ring", "gaussian"))
print()

POSITIVE_STRINGS = {""}
NEGATIVE_STRINGS = {}

# Filter
exp_handler.positive_filters = POSITIVE_STRINGS
exp_handler.negative_filters = NEGATIVE_STRINGS

exp_handler.print_structure_factor_fitted_parameters("confocal")
exp_handler.print_structure_factor_fitted_parameters("sted")

# Save
SHOULD_SAVE_FIGURES = True
# SHOULD_SAVE_FIGURES = False
save_figure(ax.figure, "StructureFactors.eps")

# Figure - Calibration ACFs & Hankel Transforms (spatial and inverse-spatial)

Load relevant experiments

In [ ]:
DATA_TYPE = "solution"

general_options = dict(
    should_load_data=False, # if False, will skip loading & processing raw data (or even checking if it exists).

    was_processed=False,

    should_parallel_process=True,

    byte_data_slice=slice(21, None, None), # ignore the first 100 elements of data - massive jumps in runtime there sometimes

    force_save = True,
    # force_save = False,
)

# "FL SD OC 1", "FL SD OC 2", "FL D OC 1", 'FL SD L 1'


data_configs = {
    "FL D Calib. 2": dict(
        confocal_date="16_04_2024",
        confocal_template="YOYO_1kbp_OLD_13uW_snake_angular_exc_054150_*.pkl",

        sted_date="16_04_2024",
        sted_template="YOYO_1kbp_OLD_13uW_snake_angular_sted_260mW_060509_*.pkl",

        force_processing=False,
        **general_options,
    ),
    "PL D Calib. 2": dict(
        confocal_date="17_04_2024",
        confocal_template="YOYO_1kbp_13uW_snake_angular_exc_141424_*.pkl",

        sted_date="17_04_2024",
        sted_template="YOYO_1kbp_13uW_snake_angular_sted_260mW_143433_*.pkl",

        force_processing=False,
        **general_options,
    ),
}

# ############################
# Load/Process the experiments
# ############################
FORCE_PROCESSING = False
# FORCE_PROCESSING = True

# PRINT_LOG_FILES = True
PRINT_LOG_FILES = False

exp_handler = SolutionSFCSExperimentHandler(
    data_root=DATA_ROOT,
    force_processing=FORCE_PROCESSING,
)
exp_handler.load_experiments(
    data_configs,
    print_log_files=PRINT_LOG_FILES
)

# Combine all but the hard-gated (SD) calibration measurements
CALIB_EXP_LABELS_TO_COMBINE = ["FL D Calib. 2", "PL D Calib. 2"]
print(f"Combining calibrations: {CALIB_EXP_LABELS_TO_COMBINE}")
combined_confocal_calib = combine_measurements_list(
    [exp_handler.exp_dict[label].confocal for label in CALIB_EXP_LABELS_TO_COMBINE if exp_handler.exp_dict.get(label) is not None]
)
combined_sted_meas = combine_measurements_list(
    [exp_handler.exp_dict[label].sted for label in CALIB_EXP_LABELS_TO_COMBINE if exp_handler.exp_dict.get(label) is not None]
)

# Create a new SolutionSFCSExperiment with the combined measurements
unified_calibration_exp = SolutionSFCSExperiment("Calibration")
unified_calibration_exp.load_experiment(confocal=combined_confocal_calib, sted=combined_sted_meas)

# Create an experiment handler just for the calibration experiment for easily re-calculating the missing Hankel transforms
figure_exp_handler = SolutionSFCSExperimentHandler(DATA_ROOT, dict(Calibration=unified_calibration_exp))
figure_exp_handler.calculate_hankel_transforms(force=False)

Actual plotting and saving of figure

In [ ]:
import matplotlib.ticker as ticker

unified_calibration_exp.name = ""

# Plot the spatial ACFs in real and inverse (q) space, side-by-side
with Plotter(super_title="Calibration, $300~bp$ DNA", subplots=(1, 2), figsize=(10, 3)) as axes:
    ax_real, ax_inverse = axes

    # plot the ACFs and their transforms
    unified_calibration_exp.plot_correlation_functions(parent_ax=ax_real, x_scale="log", y_scale="linear")
    for cf_name, cf in unified_calibration_exp.cf_dict.items():
        cf.hankel_transforms["gaussian"].plot(parent_ax=ax_inverse, plot_interpolations=False)

    # Set the axis properties for the figure
    ax_real.set_title("Spatial ACFs")
    ax_real.set_xlabel("$r,~\mu m$")
    ax_real.set_xlim(5e-3, 1.5)
    ax_real.set_ylim(0, 1.05)

    ax_inverse.set_title("Inverse-Space ACFs")
    ax_inverse.set_xlabel("$q,~\mu m^{-1}$")
    ax_inverse.set_ylabel("")
    ax_inverse.set_xlim(3, 30)
    ax_inverse.set_ylim(2e-3, 1.05)

    # move the legend to top right
    ax_real.legend(["Confocal", "STED", "gSTED"], loc="upper right")

# # Adjust x-axis tick labels for the inverse-space ACFs plot
# ax_inverse.set_xticklabels(ax_inverse.get_xticks(), rotation=45, ha='right')

# # Adjust x-axis tick labels for the inverse-space ACFs plot
# plt.setp(ax_inverse.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')


# Save
save_figure(ax_real.figure, "CalibrationRealInverse.eps")

# Figure - Afterpulsing Probability (Old vs. New)

In [ ]:
DATA_TYPE = "solution"

general_options = dict(
    should_load_data=False, # if False, will skip loading & processing raw data (or even checking if it exists).
    
    was_processed=False,
    
    should_parallel_process=True,

    afterpulsing_method="none", # Needed here to avoid attempting to remove afteroulsing (as well as avoid unecessary creation of TDC calibration)
        
    # force_save = True,
    force_save = False,
)

data_configs = {
    ###############
    # NEW #
    ###############
    "White Noise 5 kHz (new, after fix)": dict(
        confocal_date="17_08_2022",
        confocal_template="halogen5khz_static_exc_132750_*.pkl",

        force_processing=False,
        **general_options,
    ),
    "White Noise 60 kHz  (new, after fix)": dict(
        confocal_date="17_08_2022",
        confocal_template="halogen60khz_static_exc_145502_*.pkl",

        force_processing=False,
        **general_options,
    ),
    "White Noise 300 kHz  (new, after fix)": dict(
        confocal_date="17_08_2022",
        confocal_template="halogen300khz_static_exc_124749_*.pkl",

        force_processing=False,
        **general_options,
    ),
    ##########
    # OLD #
    ##########
    "White Noise 4 kHz (old detector)": dict(
        confocal_date="08_11_2021",
        confocal_template="halogen_4kHz_static_exc_112742_*.pkl",

        force_processing=False,
        **general_options,
    ),
    "White Noise 40 kHz  (old detector)": dict(
        confocal_date="08_11_2021",
        confocal_template="halogen_40kHz_static_exc_121330_*.pkl",

        force_processing=False,
        **general_options,
    ),
    "White Noise 300 kHz  (old detector)": dict(
        confocal_date="08_11_2021",
        confocal_template="halogen_300kHz_static_exc_125927_*.pkl",

        force_processing=False,
        **general_options,
    ),
}

# FORCE_PROCESSING = False
FORCE_PROCESSING = True

# PRINT_LOG_FILES = True
PRINT_LOG_FILES = False

exp_handler = SolutionSFCSExperimentHandler(
    data_root=DATA_ROOT,
    force_processing=FORCE_PROCESSING,
)
exp_handler.load_experiments(
        data_configs,
        print_log_files=PRINT_LOG_FILES
)

In [ ]:
# Limit the bottom time to the max of the minimum times of all considered measurements (done by eye from graph)
t_lims = Limits(2e-4, np.inf)

with Plotter(
    super_title="Afterpulsing Probability Density",
    xlabel="$t\\ (ms)$",
    ylabel="Probability Density (1/ms)",
    x_scale="log",
    xlim=(1.5e-4, 2e-2),
    # ylim=(-1, 1e2),
    figsize=(5, 5),
) as ax:
    for label, exp in exp_handler.exp_dict.items():
        cf = exp.confocal.cf["confocal"]
        lag = cf.lag[t_lims.valid_indices(cf.lag)]
        avg_cf_cr = cf.avg_cf_cr[t_lims.valid_indices(cf.lag)]

        # Numerical integration to get the total “counts” under the curve
        total_area = np.abs(np.trapz(avg_cf_cr, x=lag))

        print(f"{label}: {total_area:.2f}")

        # Convert original curve into a probability density
        pdf = avg_cf_cr / total_area
        # pdf = avg_cf_cr

        # Plot the PDF
        ax.plot(lag, pdf, "--" if "old" in label else "-", label=label)

    ax.legend()

# Save
save_figure(ax.figure, "OldVsNewAP.eps")